<a href="https://colab.research.google.com/github/Mebay821/health-hackathon-project/blob/main/diabetes_risk_screening.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [17]:
import pandas as pd, numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, recall_score, precision_score, confusion_matrix

URL = "https://raw.githubusercontent.com/jbrownlee/Datasets/master/pima-indians-diabetes.data.csv"
cols = ["Pregnancies","Glucose","BloodPressure","SkinThickness","Insulin","BMI","DiabetesPedigree","Age","Outcome"]
df = pd.read_csv(URL, names=cols)

print(df.shape)
print(df["Outcome"].mean())   # share of people with diabetes
df.head()

(768, 9)
0.3489583333333333


,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigree,Age,Outcome
0,6,148,72,35,0,33.6,0.627,50,1
1,1,85,66,29,0,26.6,0.351,31,0
2,8,183,64,0,0,23.3,0.672,32,1
3,1,89,66,23,94,28.1,0.167,21,0
4,0,137,40,35,168,43.1,2.288,33,1


In [18]:
for c in ["Glucose","BloodPressure","BMI"]:
    df[c] = df[c].replace(0, np.nan)
    df[c] = df[c].fillna(df[c].median())

In [19]:
X = df[["Age","BMI","BloodPressure","Glucose"]]
y = df["Outcome"]

X_trainval, X_test, y_trainval, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)
X_train, X_val, y_train, y_val = train_test_split(X_trainval, y_trainval, test_size=0.25, stratify=y_trainval, random_state=42)

In [20]:
def report(name, y_true, y_pred):
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
    print(f"{name}: accuracy {accuracy_score(y_true,y_pred):.1%}, recall {recall_score(y_true,y_pred):.1%}, "
          f"missed cases {fn}, false alarms {fp}")

# Baseline: everyone is "no diabetes"
report("Baseline", y_val, np.zeros(len(y_val)))

# First model
model = LogisticRegression(max_iter=1000, class_weight="balanced")
model.fit(X_train, y_train)
report("Model", y_val, model.predict(X_val))

Baseline: accuracy 64.9%, recall 0.0%, missed cases 54, false alarms 0
Model: accuracy 74.7%, recall 74.1%, missed cases 14, false alarms 25


In [21]:
probs = model.predict_proba(X_val)[:,1]
for t in [0.5, 0.4, 0.3, 0.2]:
    report(f"threshold {t}", y_val, (probs >= t).astype(int))

threshold 0.5: accuracy 74.7%, recall 74.1%, missed cases 14, false alarms 25
threshold 0.4: accuracy 69.5%, recall 75.9%, missed cases 13, false alarms 34
threshold 0.3: accuracy 68.2%, recall 87.0%, missed cases 7, false alarms 42
threshold 0.2: accuracy 61.7%, recall 94.4%, missed cases 3, false alarms 56


In [22]:
X = df[["Age","BMI","BloodPressure","Glucose"]]
y = df["Outcome"]

X_trainval, X_test, y_trainval, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)
X_train, X_val, y_train, y_val = train_test_split(X_trainval, y_trainval, test_size=0.25, stratify=y_trainval, random_state=42)

print(len(X_train), len(X_val), len(X_test))

460 154 154


In [23]:
def report(name, y_true, y_pred):
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
    print(f"{name}: accuracy {accuracy_score(y_true,y_pred):.1%}, recall {recall_score(y_true,y_pred):.1%}, "
          f"missed cases {fn}, false alarms {fp}")

report("Baseline", y_val, np.zeros(len(y_val)))

model = LogisticRegression(max_iter=1000, class_weight="balanced")
model.fit(X_train, y_train)
report("Model", y_val, model.predict(X_val))

Baseline: accuracy 64.9%, recall 0.0%, missed cases 54, false alarms 0
Model: accuracy 74.7%, recall 74.1%, missed cases 14, false alarms 25


In [24]:
probs = model.predict_proba(X_val)[:,1]
for t in [0.5, 0.4, 0.3, 0.2]:
    report(f"threshold {t}", y_val, (probs >= t).astype(int))

threshold 0.5: accuracy 74.7%, recall 74.1%, missed cases 14, false alarms 25
threshold 0.4: accuracy 69.5%, recall 75.9%, missed cases 13, false alarms 34
threshold 0.3: accuracy 68.2%, recall 87.0%, missed cases 7, false alarms 42
threshold 0.2: accuracy 61.7%, recall 94.4%, missed cases 3, false alarms 56


In [25]:
t = 0.3
test_probs = model.predict_proba(X_test)[:,1]
report("Baseline (test)", y_test, np.zeros(len(y_test)))
report(f"Model, threshold {t} (test)", y_test, (test_probs >= t).astype(int))

Baseline (test): accuracy 64.9%, recall 0.0%, missed cases 54, false alarms 0
Model, threshold 0.3 (test): accuracy 66.9%, recall 87.0%, missed cases 7, false alarms 44


In [26]:
test_pred = (test_probs >= 0.3).astype(int)
groups = pd.cut(X_test["Age"], bins=[0, 30, 45, 100], labels=["under 30", "30-45", "over 45"])

for g in ["under 30", "30-45", "over 45"]:
    mask = (groups == g).values
    n_sick = int(y_test[mask].sum())
    tn, fp, fn, tp = confusion_matrix(y_test[mask], test_pred[mask]).ravel()
    print(f"{g}: {mask.sum()} patients, {n_sick} sick, caught {tp}, missed {fn}, false alarms {fp}")

under 30: 89 patients, 18 sick, caught 14, missed 4, false alarms 27
30-45: 40 patients, 26 sick, caught 23, missed 3, false alarms 8
over 45: 25 patients, 10 sick, caught 10, missed 0, false alarms 9


In [27]:
val_groups = pd.cut(X_val["Age"], bins=[0, 30, 45, 100], labels=["under 30", "30-45", "over 45"])
mask = (val_groups == "under 30").values

for t in [0.3, 0.25, 0.2, 0.15]:
    pred = (probs[mask] >= t).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_val[mask], pred).ravel()
    print(f"threshold {t}: caught {tp}, missed {fn}, false alarms {fp}")

threshold 0.3: caught 11, missed 3, false alarms 18
threshold 0.25: caught 12, missed 2, false alarms 21
threshold 0.2: caught 14, missed 0, false alarms 28
threshold 0.15: caught 14, missed 0, false alarms 37


In [28]:
thr = np.where(X_test["Age"] <= 30, 0.2, 0.3)
pred_group = (test_probs >= thr).astype(int)

report("Single threshold 0.3 (test)", y_test, (test_probs >= 0.3).astype(int))
report("Per-age threshold (test)", y_test, pred_group)

mask = (X_test["Age"] <= 30).values
tn, fp, fn, tp = confusion_matrix(y_test[mask], pred_group[mask]).ravel()
print(f"Under 30 only: caught {tp}, missed {fn}, false alarms {fp}")

Single threshold 0.3 (test): accuracy 66.9%, recall 87.0%, missed cases 7, false alarms 44
Per-age threshold (test): accuracy 61.7%, recall 90.7%, missed cases 5, false alarms 54
Under 30 only: caught 16, missed 2, false alarms 37


In [29]:
def risk_check(age, bmi, blood_pressure, glucose):
    x = pd.DataFrame([[age, bmi, blood_pressure, glucose]], columns=["Age","BMI","BloodPressure","Glucose"])
    p = model.predict_proba(x)[0, 1]
    cut = 0.2 if age <= 30 else 0.3
    if p >= cut:
        return f"HIGH risk (score {p:.0%}). Refer for a blood test."
    elif p >= 0.2:
        return f"MEDIUM risk (score {p:.0%}). Recheck in a few months, or refer if symptoms."
    return f"LOW risk (score {p:.0%}). No action needed now."

print(risk_check(25, 22, 70, 95))
print(risk_check(52, 34, 85, 160))

LOW risk (score 6%). No action needed now.
HIGH risk (score 87%). Refer for a blood test.
